### Python Concurrency: GIL, Multithreading, Multiprocessing & AsyncIO

GIL - Global Interpreter Lock

It allows only one thread at a time

This does not mean Python cannot run multiple tasks concurrently.
- Threads can overlap I/O waits.
- AsyncIO can run many I/O operations concurrently in one thread.
- Multiple processes can execute Python computations on separate CPU cores.
- Free-threaded CPython builds can run Python threads without the traditional GIL, subject to compatibility and implementation details.

MultiThreading

In [1]:
import time

def fetch_data(service):
    print(f"Fetching {service}")
    time.sleep(2)
    return f"{service} completed"

start = time.perf_counter()

results = [
    fetch_data("Qdrant"),
    fetch_data("PostgreSQL"),
    fetch_data("LLM API")
]

print(results)
print(f"Elapsed: {time.perf_counter() - start:.2f}s")


Fetching Qdrant
Fetching PostgreSQL
Fetching LLM API
['Qdrant completed', 'PostgreSQL completed', 'LLM API completed']
Elapsed: 6.00s


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch_data(service):
    time.sleep(2)
    return f"{service} completed"

start = time.perf_counter()

services = ["Qdrant", "PostgreSQL", "LLM API"]

with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(fetch_data, services))

print(results)
print(f"Elapsed: {time.perf_counter() - start:.2f}s")


['Qdrant completed', 'PostgreSQL completed', 'LLM API completed']
Elapsed: 2.00s


Multiprocessing

AsyncIO

In [17]:
import asyncio
import time

# async def fetch_data(service):
#     await asyncio.sleep(2)
#     return f"{service} is completed"

# async def main():
#     start=time.perf_counter()

#     result=await asyncio.gather(
#         fetch_data("Qdrant"),
#         fetch_data("Qdrant"),
#         fetch_data("Qdrant"))
    
#     print(result)
#     # a=await fetch_data("Qdrant")
#     # b=await fetch_data("Qdrant")
#     # c=await fetch_data("Qdrant")

#     # print(a,b,c)
#     print(f"Elapsed: {time.perf_counter() - start:.2f}s")

# await main()

In [20]:
import asyncio
import time

async def call_llm(prompt):
    await asyncio.sleep(2)
    return f"Processed: {prompt}"

async def process_prompts(prompts):
    tasks = [
        call_llm(prompt)
        for prompt in prompts
    ]

    results = await asyncio.gather(*tasks)
    return results

async def main():
    prompts = [
        "Summarize invoice 1",
        "Summarize invoice 2",
        "Summarize invoice 3",
        "Summarize invoice 4",
        "Summarize invoice 5"
    ]

    start = time.perf_counter()

    result = await process_prompts(prompts)

    print(result)
    print(f"Time: {time.perf_counter() - start:.2f}s")
    
await main()


['Processed: Summarize invoice 1', 'Processed: Summarize invoice 2', 'Processed: Summarize invoice 3', 'Processed: Summarize invoice 4', 'Processed: Summarize invoice 5']
Time: 2.02s


In [21]:

import asyncio
import time

# Simulated LLM API
async def call_llm(prompt):
    print(f"Starting: {prompt}")
    await asyncio.sleep(2)
    print(f"Finished: {prompt}")
    return f"Processed: {prompt}"


async def process_prompts(prompts):
    semaphore = asyncio.Semaphore(3)

    async def limited_call(prompt):
        async with semaphore:
            return await call_llm(prompt)

    tasks = [
        limited_call(prompt)
        for prompt in prompts
    ]

    results = await asyncio.gather(*tasks)
    return results


async def main():
    prompts = [
        f"Summarize invoice {i}"
        for i in range(1, 11)
    ]

    start = time.perf_counter()

    results = await process_prompts(prompts)

    print("\nResults:", results)
    print(f"Total time: {time.perf_counter() - start:.2f}s")


# In Jupyter Notebook:
await main()

# In a normal Python script:
# asyncio.run(main())


Starting: Summarize invoice 1
Starting: Summarize invoice 2
Starting: Summarize invoice 3
Finished: Summarize invoice 1
Finished: Summarize invoice 2
Finished: Summarize invoice 3
Starting: Summarize invoice 4
Starting: Summarize invoice 5
Starting: Summarize invoice 6
Finished: Summarize invoice 4
Finished: Summarize invoice 5
Finished: Summarize invoice 6
Starting: Summarize invoice 7
Starting: Summarize invoice 8
Starting: Summarize invoice 9
Finished: Summarize invoice 7
Finished: Summarize invoice 8
Finished: Summarize invoice 9
Starting: Summarize invoice 10
Finished: Summarize invoice 10

Results: ['Processed: Summarize invoice 1', 'Processed: Summarize invoice 2', 'Processed: Summarize invoice 3', 'Processed: Summarize invoice 4', 'Processed: Summarize invoice 5', 'Processed: Summarize invoice 6', 'Processed: Summarize invoice 7', 'Processed: Summarize invoice 8', 'Processed: Summarize invoice 9', 'Processed: Summarize invoice 10']
Total time: 8.02s
